# Air Quality Analysis
**Tasks 49–63**

Student: Kumar Mangalam Yaduvanshi  
Admission No.: 2026b01010939  
Repository: https://github.com/kmy2106/HCL-.git

This notebook uses **Pandas and NumPy** (plus Python's built-in `pathlib`). PM2.5 categories and threshold are project-selected, not official AQI standards.

## Setup
Ensure `mayank_air_quality_1000.csv` is in the notebook's working directory. If needed, install dependencies with `pip install pandas numpy`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# STUDENT INFORMATION

In [ ]:
NAME = "Kumar Mangalam Yaduvanshi"
ADMISSION_NO = "2026b01010939"
REPO_LINK = "https://github.com/kmy2106/HCL-.git"

print("Student:", NAME)
print("Admission No.:", ADMISSION_NO)
print("Repository:", REPO_LINK)

In [ ]:
# FILE SETUP

In [ ]:
# In a notebook, this uses the folder containing the notebook's working directory.
BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "mayank_air_quality_1000.csv"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}\n"
        "Place the CSV file in the notebook's working directory."
    )

In [ ]:
# ANALYSIS SETTINGS

In [ ]:
# Illustrative project categories, NOT official AQI classifications.
CATEGORY_BINS = [-np.inf, 15, 35, 55, np.inf]
CATEGORY_LABELS = [
    "Low (chosen)", "Moderate (chosen)",
    "Elevated (chosen)", "High (chosen)"
]
PM25_ABOVE_THRESHOLD = 60  # Project-selected analytical threshold

In [ ]:
# TASK 49 - DATASET INSPECTION

In [ ]:
df = pd.read_csv(DATA_FILE)

print("=" * 70)
print("TASK 49 - DATASET INSPECTION")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(10))
print("\\nData types:\\n", df.dtypes)
display(df.describe(include="all"))

In [ ]:
# COLUMN VALIDATION

In [ ]:
required = ["Date", "City", "PM25", "PM10", "Temperature_C"]
missing_cols = [col for col in required if col not in df.columns]
if missing_cols:
    raise ValueError(f"Expected columns missing from dataset: {missing_cols}")

# NO2 is optional; if absent, create it so Task 55 can proceed.
if "NO2" not in df.columns:
    df["NO2"] = np.nan

In [ ]:
# NUMERIC CONVERSION

In [ ]:
for col in ["PM25", "PM10", "Temperature_C", "NO2"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# TASK 50 - MISSING VALUES BEFORE CLEANING

In [ ]:
missing_summary = df.isna().sum().to_frame("Missing_Count")
missing_summary["Missing_Percent"] = (
    missing_summary["Missing_Count"] / len(df) * 100
).round(2)
display(missing_summary)
missing_summary.to_csv(OUT_DIR / "missing_values.csv")

In [ ]:
# TASK 51 - FILL MISSING PM25 VALUES

In [ ]:
pm25_missing_before = int(df["PM25"].isna().sum())
overall_median = df["PM25"].median()
if pd.isna(overall_median):
    raise ValueError("PM25 column contains no valid numeric values.")

city_medians = df.groupby("City")["PM25"].transform("median")
df["PM25"] = df["PM25"].fillna(city_medians).fillna(overall_median)
print(f"Filled {pm25_missing_before} missing PM25 values.")

In [ ]:
# TASK 52 - DATE PARSING AND MONTH EXTRACTION

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
invalid_dates = int(df["Date"].isna().sum())
df = df.dropna(subset=["Date"]).copy()
if df.empty:
    raise ValueError("No valid records remain after date cleaning.")

df["Month"] = df["Date"].dt.strftime("%Y-%m")
df["Date_Only"] = df["Date"].dt.date.astype(str)
print("Invalid dates removed:", invalid_dates)

In [ ]:
# TASK 53 - PM25 SUMMARY

In [ ]:
pm25_stats = pd.Series({
    "Mean": df["PM25"].mean(),
    "Median": df["PM25"].median(),
    "Maximum": df["PM25"].max()
}).round(2)
display(pm25_stats)
pm25_stats.to_csv(OUT_DIR / "pm25_summary.csv", header=["Value"])

In [ ]:
# TASK 54 - CITY RANKING

In [ ]:
city_ranking = (
    df.groupby("City")["PM25"].mean()
    .sort_values(ascending=False).round(2)
    .rename("Average_PM25").reset_index()
)
display(city_ranking)
city_ranking.to_csv(OUT_DIR / "city_ranking.csv", index=False)

In [ ]:
# TASK 55 - TOP 20 PM25 OBSERVATIONS

In [ ]:
columns_to_show = ["Date", "City", "PM25", "PM10", "Temperature_C"]
if "NO2" in df.columns:
    columns_to_show.insert(4, "NO2")
high_dates = df.nlargest(20, "PM25")[columns_to_show]
display(high_dates)
high_dates.to_csv(OUT_DIR / "highest_readings.csv", index=False)

In [ ]:
# TASK 56 - CITY-WISE PM10 AND PM25 COMPARISON

In [ ]:
pollutant_comparison = df.groupby("City")[["PM25", "PM10"]].mean().round(2)
pollutant_comparison["PM10_to_PM25_Ratio"] = (
    pollutant_comparison["PM10"] /
    pollutant_comparison["PM25"].replace(0, np.nan)
).round(2)
pollutant_comparison = pollutant_comparison.reset_index()
display(pollutant_comparison)
pollutant_comparison.to_csv(OUT_DIR / "pm10_pm25_comparison.csv", index=False)

In [ ]:
# TASK 57 - PM25 CATEGORIES (PROJECT-DEFINED)

In [ ]:
df["PM25_Category"] = pd.cut(
    df["PM25"], bins=CATEGORY_BINS, labels=CATEGORY_LABELS, right=True
)
category_counts = df["PM25_Category"].value_counts(sort=False).to_frame("Count")
category_counts["Percent"] = (category_counts["Count"] / len(df) * 100).round(2)
display(category_counts)
category_counts.to_csv(OUT_DIR / "category_counts.csv")

In [ ]:
# TASK 58 - CITY/MONTH AVERAGES

In [ ]:
monthly_avg = (
    df.groupby(["City", "Month"])["PM25"]
    .mean().round(2).reset_index(name="Average_PM25")
)
display(monthly_avg.head(30))
monthly_avg.to_csv(OUT_DIR / "monthly_city_average.csv", index=False)

In [ ]:
# TASK 59 - STANDARD DEVIATION AND PERCENTILES

In [ ]:
values = df["PM25"].to_numpy(dtype=float)
std_dev = float(np.std(values, ddof=0))
percentiles = {f"P{p}": float(np.percentile(values, p))
               for p in [25, 50, 75, 90, 95, 99]}
stats_59 = pd.DataFrame([{"Standard_Deviation": std_dev, **percentiles}]).round(2)
display(stats_59)
stats_59.to_csv(OUT_DIR / "std_percentiles.csv", index=False)

In [ ]:
# TASK 60 - IQR OUTLIERS

In [ ]:
q1, q3 = np.percentile(values, [25, 75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[
    (df["PM25"] < lower_bound) | (df["PM25"] > upper_bound)
].sort_values("PM25", ascending=False)
display(outliers[["Date", "City", "PM25"]])
print(f"IQR bounds: {lower_bound:.2f}, {upper_bound:.2f}")
outliers[["Date", "City", "PM25"]].to_csv(OUT_DIR / "iqr_outliers.csv", index=False)

In [ ]:
# TASK 61 - TEMPERATURE/PM25 CORRELATION

In [ ]:
correlation = df[["Temperature_C", "PM25"]].corr().iloc[0, 1]
print("Pearson correlation:", round(float(correlation), 4))
pd.DataFrame([{"Pearson_Correlation_Temperature_PM25": correlation}]).to_csv(
    OUT_DIR / "temperature_correlation.csv", index=False
)

In [ ]:
# TASK 62 - ABOVE PROJECT-SELECTED THRESHOLD

In [ ]:
above = df["PM25"] > PM25_ABOVE_THRESHOLD
threshold_summary = pd.DataFrame([{
    "Chosen_Threshold_PM25": PM25_ABOVE_THRESHOLD,
    "Count_Above": int(above.sum()),
    "Total_Observations": len(df),
    "Percent_Above": round(float(above.mean() * 100), 2)
}])
display(threshold_summary)
threshold_summary.to_csv(OUT_DIR / "above_threshold_summary.csv", index=False)

In [ ]:
# TASK 63 - FIVE DATA-INFORMED RECOMMENDATIONS

In [ ]:
top_city = city_ranking.iloc[0]["City"] if not city_ranking.empty else "unknown"
top_city_mean = city_ranking.iloc[0]["Average_PM25"] if not city_ranking.empty else np.nan
peak = df.loc[df["PM25"].idxmax()]
top_months = df.groupby("Month")["PM25"].mean().sort_values(ascending=False)
peak_month = top_months.index[0] if len(top_months) else "unknown"
peak_month_mean = top_months.iloc[0] if len(top_months) else np.nan
above_pct = float(above.mean() * 100)

if pd.isna(correlation):
    corr_strength = "undetermined"
elif correlation > 0:
    corr_strength = "positive"
elif correlation < 0:
    corr_strength = "negative"
else:
    corr_strength = "near-zero"

recommendations = [
    f"Prioritize monitoring in {top_city}, with the highest city average PM2.5 ({top_city_mean:.2f}).",
    f"Plan intensified monitoring and public messaging for {peak_month}, the month with the highest average PM2.5 ({peak_month_mean:.2f}).",
    f"Consider alerts for the project's selected threshold of {PM25_ABOVE_THRESHOLD}; {above_pct:.2f}% of observations exceed it. This is not an official standard.",
    f"Investigate the peak reading ({peak['PM25']:.2f}) on {peak['Date'].strftime('%Y-%m-%d')} in {peak['City']} for possible local factors.",
    f"Temperature and PM2.5 show a {corr_strength} relationship (r={correlation:.3f}); this is descriptive and does not establish causation."
]
with open(OUT_DIR / "recommendations.txt", "w", encoding="utf-8") as f:
    f.write("FIVE DATA-INFORMED RECOMMENDATIONS\n\n")
    for i, item in enumerate(recommendations, 1):
        f.write(f"{i}. {item}\n\n")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

In [ ]:
# SAVE CLEANED DATASET AND PROJECT INFORMATION

In [ ]:
df.to_csv(OUT_DIR / "cleaned_air_quality.csv", index=False)
print("\nStudent:", NAME)
print("Admission Number:", ADMISSION_NO)
print("GitHub:", REPO_LINK)
print("\nAnalysis complete. Output files saved in:", OUT_DIR)